# 💬 chatlens — WhatsApp Chat Intelligence: end-to-end Data Science pipeline (Google Colab)

This notebook runs the **whole project** on Colab and saves everything to **Google Drive**:
data download (Kaggle) → cleaning & leak-free splits → **model benchmarks** (sentiment, emotion) →
**topic & keyword method comparisons** → **statistical analysis** → analysis of a WhatsApp chat →
reports, model export, experiment tracking, and a downloadable deliverables bundle.

**How to use**
1. Runtime ▸ *Change runtime type* ▸ **T4 GPU** (needed only for the transformer methods; everything else runs on CPU).
2. Put your Kaggle token in Colab ▸ 🔑 **Secrets** as `KAGGLE_API_TOKEN` (Kaggle ▸ Settings ▸ API ▸ *Create New Token*).
   Grant the notebook access. (Legacy `KAGGLE_USERNAME` + `KAGGLE_KEY` secrets also work.)
3. Run cells top to bottom. **If the runtime disconnects, just re-run from the top**: finished stages are verified
   (SHA-256) on Drive and skipped; only the interrupted stage repeats (fine-tuning resumes from its last epoch).
4. To analyse **your own chat**, export it from WhatsApp (*Without media*), put the `.txt`/`.zip` in
   `Drive/chatlens_project/chats/` and set `CHAT_FILE` below. It never leaves your Drive/this runtime.

**Honesty notes**: no result is shown unless it was computed in this run; the Kaggle corpora are *not* WhatsApp
conversations (see the dataset table); a chat produced by the built-in generator is **fabricated** and labelled as such.

## 0 · Configuration (edit here only)

In [1]:
import json, os
CONFIG = dict(
    DRIVE_ROOT="/content/drive/MyDrive/chatlens_project",
    REPO_URL="https://github.com/anil-67140/DS_chatlens.git", REPO_BRANCH="main",
    SEED=42,
    MODE="full",                    # "full" | "smoke" (tiny/fast: for checking the plumbing, NOT for reporting results)
    RUN_TRANSFORMERS=True,          # pretrained + fine-tuned transformer methods, sentence embeddings, BERTopic
    FINETUNE_EPOCHS=3, FINETUNE_MAX_TRAIN=20000,   # cap fine-tuning set size to bound GPU time
    SENT140_ROWS=100_000,           # Sentiment140 subsample (1.6M rows total; labels are emoticon-derived = noisy)
    ROMAN_URDU_MAX_ROWS=None,
    TOPIC_MAX_DOCS=20_000, TOPIC_KS=(5, 10, 15, 20),
    TOPIC_METHODS=("lda", "nmf", "kmeans_tfidf", "lsa_kmeans", "embedding_kmeans", "bertopic"),
    N_BOOT=1000,
    SMOKE_METHODS=None,             # list of sentiment method names to restrict to (smoke tests only)
    CHAT_FILE=None,                 # e.g. "WhatsApp Chat with X.txt" inside DRIVE_ROOT/chats ; None => fabricated demo chat
    ANONYMIZE=True,
    DATASET_SOURCE="kaggle",        # "kaggle" (default). Roman Urdu falls back to its GitHub mirror if Kaggle fails.
    LAUNCH_DASHBOARD=False,         # open the Gradio dashboard at the end (share=True makes a PUBLIC link: see warning)
)
# test hook: CHATLENS_NB_OVERRIDES='{"MODE": "smoke"}' (used by the automated notebook run; ignore in Colab)
CONFIG.update(json.loads(os.environ.get("CHATLENS_NB_OVERRIDES", "{}")))
SMOKE = CONFIG["MODE"] == "smoke"
print({k: v for k, v in CONFIG.items()})

{'DRIVE_ROOT': '/content/drive/MyDrive/chatlens_project', 'REPO_URL': 'https://github.com/anil-67140/DS_chatlens.git', 'REPO_BRANCH': 'main', 'SEED': 42, 'MODE': 'full', 'RUN_TRANSFORMERS': True, 'FINETUNE_EPOCHS': 3, 'FINETUNE_MAX_TRAIN': 20000, 'SENT140_ROWS': 100000, 'ROMAN_URDU_MAX_ROWS': None, 'TOPIC_MAX_DOCS': 20000, 'TOPIC_KS': (5, 10, 15, 20), 'TOPIC_METHODS': ('lda', 'nmf', 'kmeans_tfidf', 'lsa_kmeans', 'embedding_kmeans', 'bertopic'), 'N_BOOT': 1000, 'SMOKE_METHODS': None, 'CHAT_FILE': None, 'ANONYMIZE': True, 'DATASET_SOURCE': 'kaggle', 'LAUNCH_DASHBOARD': False}


## 1 · Mount Google Drive & create the project folders

In [2]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    CONFIG["DRIVE_ROOT"] = os.environ.get("CHATLENS_LOCAL_ROOT", CONFIG["DRIVE_ROOT"] if os.path.isabs(CONFIG["DRIVE_ROOT"]) and os.access(os.path.dirname(CONFIG["DRIVE_ROOT"]) or "/", os.W_OK) else "./chatlens_project")
ROOT = Path(CONFIG["DRIVE_ROOT"]); ROOT.mkdir(parents=True, exist_ok=True)
for sub in ["data/raw", "data/processed", "models", "reports/tables", "reports/figures", "reports/chat",
            "logs", "chats", "nlp", "topics", "keywords", "deliverables"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Private chats folder (never exported):", ROOT / "chats")

Mounted at /content/drive
Project root: /content/drive/MyDrive/chatlens_project
Private chats folder (never exported): /content/drive/MyDrive/chatlens_project/chats


## 2 · Install the project (all packages are installed BEFORE anything is imported)
Installs `chatlens` from your GitHub repo (the version containing this notebook must be pushed), or from a
`chatlens.zip` you place in the Drive project folder (takes priority). Installing everything first matters: upgrading
numpy/pandas after they were imported crashes a running kernel. If this cell reports that core libraries changed,
**Runtime ▸ Restart session, then run again** (finished stages are not repeated).

In [3]:
import importlib, importlib.util, importlib.metadata as md_, subprocess, shutil, zipfile
CORE = ["numpy", "pandas", "scipy", "scikit-learn"]
def _ver(p):
    try: return md_.version(p)
    except md_.PackageNotFoundError: return None
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(f"command failed: {cmd}")
    return r
def _chatlens_ok():
    v = _ver("chatlens")
    return v is not None and tuple(map(int, v.split(".")[:2])) >= (0, 2)
before = {p: _ver(p) for p in CORE}
if not _chatlens_ok():
    if (ROOT / "chatlens.zip").exists():
        src = Path("/content/chatlens_src"); shutil.rmtree(src, ignore_errors=True)
        with zipfile.ZipFile(ROOT / "chatlens.zip") as z: z.extractall(src)
        src = next(src.rglob("pyproject.toml")).parent
    else:
        src = Path("/content/DS_chatlens"); shutil.rmtree(src, ignore_errors=True)
        sh(f"git clone --depth 1 -b {CONFIG['REPO_BRANCH']} {CONFIG['REPO_URL']} {src}")
    # NOTE: a normal (non-editable) install: an editable install is not importable from the already-running kernel.
    sh(f"{sys.executable} -m pip install -q '{src}[app,tracking,kaggle]'")
    SRC_DIR = src
if not SMOKE:     # Colab preinstalls an old kaggle CLI that does not understand KAGGLE_API_TOKEN
    sh(f"{sys.executable} -m pip install -q -U 'kaggle>=2.2'", check=False)
else:
    SRC_DIR = None
if CONFIG["RUN_TRANSFORMERS"] and not SMOKE:
    sh(f"{sys.executable} -m pip install -q transformers sentence-transformers bertopic hdbscan umap-learn spacy", check=False)
    sh(f"{sys.executable} -m spacy download en_core_web_sm -q", check=False)
sh(f"{sys.executable} -m pip freeze > '{ROOT / 'logs' / 'environment.lock.txt'}'")     # exact package versions, on Drive
after = {p: _ver(p) for p in CORE}
changed = {p: (before[p], after[p]) for p in CORE if before[p] != after[p] and p in sys.modules}
if changed:
    raise SystemExit(f"Core libraries changed while loaded: {changed}\n=> Runtime > Restart session, then run all cells again.")
importlib.invalidate_caches()
if importlib.util.find_spec("chatlens") is None and SRC_DIR is not None:      # last resort: use the source tree directly
    sys.path.insert(0, str(SRC_DIR / "src")); importlib.invalidate_caches()
import chatlens
print("chatlens", chatlens.__version__, "from", Path(chatlens.__file__).parent)
assert tuple(map(int, chatlens.__version__.split(".")[:2])) >= (0, 2), (
    "Your GitHub repo still has the OLD chatlens. Push the new code, or upload chatlens.zip to the Drive project folder.")
import platform
try:
    import torch; GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"
except Exception: GPU = "torch not installed"
print("Python", platform.python_version(), "| GPU:", GPU)
if CONFIG["RUN_TRANSFORMERS"] and GPU.startswith("none"):
    print("⚠️  No GPU: fine-tuning will be very slow. Switch the runtime to GPU or set RUN_TRANSFORMERS=False.")

chatlens 0.2.0 from /usr/local/lib/python3.13/dist-packages/chatlens
Python 3.13.16 | GPU: Tesla T4


## 3 · Reproducibility, checkpoint store & experiment tracking

In [4]:
import logging, random, time
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
from chatlens.checkpoint import CheckpointStore
from chatlens.tracking import Tracker, environment_info
from chatlens.logging_utils import get_logger

SEED = CONFIG["SEED"]
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
except Exception: pass

store = CheckpointStore(ROOT)                                    # ONE manifest for the whole project (Drive)
tracker = Tracker(ROOT / "mlflow", "chatlens", local_dir="/tmp/mlflow_live")   # SQLite lives locally, synced to Drive
fh = logging.FileHandler(ROOT / "logs" / "run.log"); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
get_logger().addHandler(fh)                                       # structured log (counts/timings only: never message text)
print("tracking backend:", tracker.backend, "| env:", environment_info())
display(pd.DataFrame(store.summary()) if store.stages else "fresh project: no completed stages yet")

2026/11/03 13:18:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/03 13:18:27 INFO mlflow.store.db.utils: Updating database tables


tracking backend: mlflow | env: {'python': '3.13.16', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'chatlens': '0.2.0', 'pandas': '2.2.3', 'numpy': '2.4.6', 'scikit-learn': '1.6.1', 'scipy': '1.16.3', 'torch': '2.11.0+cu130', 'transformers': '5.22.0', 'mlflow': '3.17.2'}


'fresh project: no completed stages yet'

## 4 · Datasets: Kaggle download (resumable) and provenance

| Key | Kaggle dataset | Used for | Size / labels | Caveat |
|---|---|---|---|---|
| `roman_urdu` | `smat26/sentiment-analysis-on-roman-urdu` | **Sentiment** for Roman-Urdu/Hinglish chat | ~20k sentences, pos/neu/neg, manually tagged | e-commerce/social text; GPL-3.0 mirror; has duplicates + a label typo (handled) |
| `emotions` | `praveengovi/emotions-dataset-for-nlp` | **Emotion** (English, 6 classes) | 16k/2k/2k official splits | English tweets only |
| `goemotions` | `shivamb/go-emotions-google-emotions-dataset` | **Topics/keywords** corpus (informal short text) | ~58k Reddit comments | English; labels unused here |
| `sentiment140` | `kazanova/sentiment140` | **English sentiment** | 1.6M tweets (subsampled) | labels derived from emoticons = **noisy** |

None of these is a WhatsApp corpus. Real WhatsApp chats are private; public "WhatsApp datasets" are synthetic or
personal data of unknown consent, so they are deliberately **not** used. You must also accept each dataset's terms on kaggle.com.

In [5]:
from chatlens.kaggle_data import configure_kaggle_auth, download_dataset, CANDIDATES
def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
try:
    method = configure_kaggle_auth(token=_secret("KAGGLE_API_TOKEN"), username=_secret("KAGGLE_USERNAME"), key=_secret("KAGGLE_KEY"))
    print("Kaggle credentials found via:", method)
    KAGGLE_OK = True
except RuntimeError as e:
    print("⚠️ ", e, "\nKaggle downloads will be skipped (Roman Urdu falls back to its GitHub mirror).")
    KAGGLE_OK = False

Kaggle credentials found via: legacy_key


In [6]:
RAW = ROOT / "data" / "raw"
DOWNLOADS = {}
def _download(key):
    def work():
        if KAGGLE_OK and CONFIG["DATASET_SOURCE"] == "kaggle":
            r = download_dataset(key, RAW)
            DOWNLOADS[key] = {"source": "kaggle", "slug": r.slug, "files": r.files}
        else:
            raise RuntimeError("kaggle unavailable")
        store.save_json(f"data/raw/{key}/_provenance.json", DOWNLOADS[key])
    try:
        store.run(f"download:{key}", work, outputs=[f"data/raw/{key}/_COMPLETE.json", f"data/raw/{key}/_provenance.json"],
                  params={"slugs": CANDIDATES[key]})
        DOWNLOADS[key] = store.load_json(f"data/raw/{key}/_provenance.json")
    except Exception as e:
        DOWNLOADS[key] = {"source": None, "error": str(e)[:300]}
        print(f"  ✗ {key}: {str(e)[:200]}")

def _mirror_roman_urdu():
    """Fallback: the dataset author's public GitHub repo (GPL-3.0). Same data, different host."""
    dest = RAW / "roman_urdu_github"
    def work():
        tmp = RAW / "roman_urdu_github.partial"; shutil.rmtree(tmp, ignore_errors=True)
        sh(f"git clone --depth 1 https://github.com/Smat26/Roman-Urdu-Dataset.git '{tmp}'")
        shutil.rmtree(dest, ignore_errors=True); os.replace(tmp, dest)
        store.save_json("data/raw/roman_urdu_github/_provenance.json", {"source": "github_mirror", "repo": "Smat26/Roman-Urdu-Dataset", "license": "GPL-3.0"})
    store.run("download:roman_urdu_github", work, outputs=["data/raw/roman_urdu_github/_provenance.json"], params={})
    return dest

for key in ["roman_urdu", "emotions", "goemotions", "sentiment140"]:
    if SMOKE and key != "roman_urdu": DOWNLOADS[key] = {"source": None, "error": "skipped in smoke mode"}; continue
    _download(key)
if DOWNLOADS["roman_urdu"]["source"] is None:
    print("Roman Urdu: Kaggle unavailable -> using the GitHub mirror")
    _mirror_roman_urdu(); DOWNLOADS["roman_urdu"] = {"source": "github_mirror", "path": "roman_urdu_github"}
display(pd.DataFrame([{"dataset": k, **{kk: (str(vv)[:90]) for kk, vv in v.items() if kk != "files"}} for k, v in DOWNLOADS.items()]))

2026-11-03 13:18:48,639 INFO chatlens stage=download:roman_urdu RUNNING (never run)
2026-11-03 13:18:51,398 INFO chatlens downloaded smat26/roman-urdu-dataset (9 files)
2026-11-03 13:18:51,423 INFO chatlens stage=download:roman_urdu COMPLETE seconds=2.762 artifacts=2
2026-11-03 13:18:51,426 INFO chatlens stage=download:emotions RUNNING (never run)
2026-11-03 13:18:53,003 INFO chatlens downloaded praveengovi/emotions-dataset-for-nlp (3 files)
2026-11-03 13:18:53,028 INFO chatlens stage=download:emotions COMPLETE seconds=1.58 artifacts=2
2026-11-03 13:18:53,033 INFO chatlens stage=download:goemotions RUNNING (never run)
2026-11-03 13:18:54,927 INFO chatlens downloaded shivamb/go-emotions-google-emotions-dataset (1 files)
2026-11-03 13:18:54,952 INFO chatlens stage=download:goemotions COMPLETE seconds=1.899 artifacts=2
2026-11-03 13:18:54,956 INFO chatlens stage=download:sentiment140 RUNNING (never run)
2026-11-03 13:19:01,261 INFO chatlens downloaded kazanova/sentiment140 (1 files)
2026-

,dataset,slug,source
0,roman_urdu,smat26/roman-urdu-dataset,kaggle
1,emotions,praveengovi/emotions-dataset-for-nlp,kaggle
2,goemotions,shivamb/go-emotions-google-emotions-dataset,kaggle
3,sentiment140,kazanova/sentiment140,kaggle


## 5 · Data quality, cleaning and leak-free splits
Every dropped row is accounted for. Duplicates are removed **before** splitting (otherwise the same sentence lands in
train and test and inflates every score); texts with conflicting labels are removed entirely. Selection of
hyper-parameters uses the validation split; the test split is touched once, for the final numbers.

In [7]:
from chatlens.kaggle_data import load_roman_urdu, load_emotions, load_goemotions_text, load_sentiment140
from chatlens.experiments import prepare_dataset
def _root(key):
    d = DOWNLOADS.get(key, {})
    if d.get("source") == "kaggle": return RAW / key
    if d.get("source") == "github_mirror": return RAW / "roman_urdu_github"
    return None
DATA = {}
if _root("roman_urdu"):
    DATA["roman_urdu"] = prepare_dataset(store, "roman_urdu", lambda: load_roman_urdu(_root("roman_urdu")), seed=SEED,
                                         max_rows=CONFIG["ROMAN_URDU_MAX_ROWS"] or (4000 if SMOKE else None))
if _root("emotions"):
    DATA["emotions"] = prepare_dataset(store, "emotions", lambda: load_emotions(_root("emotions")), presplit=True, seed=SEED)
if _root("sentiment140"):
    DATA["sentiment140"] = prepare_dataset(store, "sentiment140", lambda: load_sentiment140(_root("sentiment140"), CONFIG["SENT140_ROWS"] * 1, SEED),
                                           seed=SEED, max_rows=CONFIG["SENT140_ROWS"])
rows = []
for k, d in DATA.items():
    q = d["quality"]
    rows.append({"dataset": k, **q["sizes"], "fingerprint": q["fingerprint"],
                 "dropped": ("see per_split" if q.get("presplit") else q["n_raw"] - q["n_clean"]),
                 "class_counts": q.get("class_counts") or {s: p["class_counts"] for s, p in q["per_split"].items()}})
display(pd.DataFrame(rows))
for k, d in DATA.items():
    if not d["quality"].get("presplit"):
        q = d["quality"]; print(f"{k}: raw {q['n_raw']:,} | missing text -{q['dropped_missing_text']} | conflicting labels -{q['dropped_conflicting']} | duplicates -{q['dropped_exact_dup']} | label typos repaired {q['labels_repaired']} | clean {q['n_clean']:,} | imbalance {q['imbalance_ratio']:.2f}:1")
        for n in q["notes"]: print("   note:", n)

2026-11-03 13:19:02,838 INFO chatlens stage=prepare:roman_urdu RUNNING (never run)
2026-11-03 13:19:03,546 INFO chatlens stage=prepare:roman_urdu COMPLETE seconds=0.683 artifacts=4
2026-11-03 13:19:03,617 INFO chatlens stage=prepare:emotions RUNNING (never run)
2026-11-03 13:19:04,702 INFO chatlens stage=prepare:emotions COMPLETE seconds=1.062 artifacts=4
2026-11-03 13:19:04,791 INFO chatlens stage=prepare:sentiment140 RUNNING (never run)
2026-11-03 13:19:11,839 INFO chatlens stage=prepare:sentiment140 COMPLETE seconds=7.011 artifacts=4


,dataset,test,train,val,fingerprint,dropped,class_counts
0,roman_urdu,3875,13559,1938,c04c1569c547b59b,857,"{'negative': 5231, 'neutral': 8347, 'positive'..."
1,emotions,1986,15939,1991,0e71bf90ea445743,see per_split,"{'test': {'anger': 275, 'fear': 224, 'joy': 69..."
2,sentiment140,19532,68358,9766,ee3a4e8679a0a75a,2344,"{'negative': 49031, 'positive': 48625}"


roman_urdu: raw 20,229 | missing text -140 | conflicting labels -154 | duplicates -520 | label typos repaired 1 | clean 19,372 | imbalance 1.60:1
sentiment140: raw 100,000 | missing text -0 | conflicting labels -571 | duplicates -1568 | label typos repaired 0 | clean 97,656 | imbalance 1.01:1


## 6 · Exploratory analysis of the training corpora

In [8]:
fig_paths = []
for name, d in DATA.items():
    tr = d["splits"]["train"]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    tr["label"].value_counts().plot.bar(ax=ax[0], color="#4c78a8", title=f"{name}: class balance (train)"); ax[0].set_xlabel("")
    tr["text"].str.split().str.len().clip(upper=60).plot.hist(bins=30, ax=ax[1], title="words per text (clipped at 60)", color="#72b7b2")
    fig.tight_layout(); p = store.save_figure(f"reports/figures/eda_{name}.png", fig); plt.close(fig); fig_paths.append(str(p))
    display(Markdown(f"**{name}** — train {len(tr):,}, median {int(tr['text'].str.split().str.len().median())} words"))
    plt.figure(figsize=(10, 3.4)); plt.imshow(plt.imread(p)); plt.axis("off"); plt.show()
print("figures saved:", len(fig_paths))

**roman_urdu** — train 13,559, median 10 words

**emotions** — train 15,939, median 17 words

**sentiment140** — train 68,358, median 12 words

figures saved: 3


## 7 · Sentiment benchmark (≥ 10 methods, compared on the *same* task)
Families: majority baseline · lexicons (VADER, TextBlob, zero-shot + tuned neutral band) · TF-IDF models (word/char/both ×
LR, calibrated SVM, NB, SGD, RF) · pretrained transformers (zero-shot) · fine-tuned transformers.
**Metrics**: accuracy, macro/weighted F1 with bootstrap CI, per-class P/R/F1, MCC, balanced accuracy, ROC-AUC & PR-AUC
(one-vs-rest) and calibration (ECE, Brier, log-loss) **only for methods that output probabilities**. Lexicons have no
probabilities, so those metrics are *not defined* for them and are left empty (never invented).

In [9]:
from chatlens.nlp.classifiers import sentiment_registry, emotion_registry
from chatlens.experiments import run_classification_task
from chatlens.viz import confusion_figure, benchmark_figure
CKPT = str(ROOT / "nlp" / "ft_ckpt")               # fine-tuning checkpoints live on Drive (resume after disconnect)
TASKS = {}
def _registry(kind):
    reg = (sentiment_registry if kind == "sentiment" else emotion_registry)(
        SEED, CKPT, include_hf=CONFIG["RUN_TRANSFORMERS"] and not SMOKE,
        finetune_epochs=CONFIG["FINETUNE_EPOCHS"], finetune_max_train=CONFIG["FINETUNE_MAX_TRAIN"])
    if CONFIG["SMOKE_METHODS"]: reg = [m for m in reg if m.name in CONFIG["SMOKE_METHODS"]]
    return reg
def run_task(task, dataset, kind):
    if dataset not in DATA:
        print(f"⏭  {task}: dataset '{dataset}' unavailable -> skipped"); return None
    out = run_classification_task(task, DATA[dataset], _registry(kind), store, tracker, SEED, CONFIG["N_BOOT"])
    TASKS[task] = out
    if out["errors"]: print("methods that failed (others continue):", out["errors"])
    if len(out["table"]):
        fig = benchmark_figure(out["table"], "macro_f1", f"{task}: macro-F1 by method (95% CI)")
        store.save_figure(f"reports/figures/{task}_macro_f1.png", fig); plt.close(fig)
        best = out["table"].iloc[0]["method"]
        fig = confusion_figure(out["metrics"][best], title=f"{task}: {best}")
        store.save_figure(f"reports/figures/{task}_confusion_best.png", fig); plt.close(fig)
        display(Markdown(f"### {task}  —  best: `{best}`"))
        display(out["table"].round(3))
        display(Markdown("**Is the best really better?** (paired McNemar with Holm correction + paired-bootstrap CI of the macro-F1 difference)"))
        display(out["significance"][["method", "macro_f1_diff", "diff_ci_low", "diff_ci_high", "holm_p", "verdict"]].round(4))
    return out
run_task("sentiment_roman_urdu", "roman_urdu", "sentiment")
tracker.sync()

2026-11-03 13:19:16,013 INFO chatlens stage=sentiment_roman_urdu:majority_baseline RUNNING (never run)
2026-11-03 13:19:16,409 INFO chatlens stage=sentiment_roman_urdu:majority_baseline COMPLETE seconds=0.366 artifacts=2
2026-11-03 13:19:16,410 INFO chatlens stage=sentiment_roman_urdu:vader RUNNING (never run)
2026-11-03 13:19:17,011 INFO chatlens stage=sentiment_roman_urdu:vader COMPLETE seconds=0.573 artifacts=2
2026-11-03 13:19:17,013 INFO chatlens stage=sentiment_roman_urdu:vader_tuned RUNNING (never run)
2026-11-03 13:19:18,139 INFO chatlens stage=sentiment_roman_urdu:vader_tuned COMPLETE seconds=1.084 artifacts=2
2026-11-03 13:19:18,141 INFO chatlens stage=sentiment_roman_urdu:textblob RUNNING (never run)
2026-11-03 13:19:20,402 INFO chatlens stage=sentiment_roman_urdu:textblob COMPLETE seconds=2.237 artifacts=2
2026-11-03 13:19:20,404 INFO chatlens stage=sentiment_roman_urdu:textblob_tuned RUNNING (never run)
2026-11-03 13:19:23,805 INFO chatlens stage=sentiment_roman_urdu:textb

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

... (46 outputs left out)

,rank,method,family,zero_shot,accuracy,macro_f1,macro_f1_ci_low,macro_f1_ci_high,weighted_f1,balanced_acc,mcc,roc_auc_ovr,pr_auc,ece,brier,fit_s,ms_per_1k
0,1,tfidf_word+char_logreg,tfidf,False,0.670,0.664,0.649,0.678,0.670,0.663,0.493,0.830,0.725,0.015,0.450,48.707,460.95
1,2,tfidf_char_logreg,tfidf,False,0.657,0.651,0.636,0.665,0.657,0.650,0.474,0.823,0.711,0.017,0.462,31.333,199.69
2,3,tfidf_word+char_linearsvc_calibrated,tfidf,False,0.663,0.649,0.634,0.663,0.659,0.641,0.477,0.832,0.729,0.050,0.452,16.434,304.37
3,4,tfidf_word+char_sgd_modhuber,tfidf,False,0.652,0.645,0.630,0.660,0.651,0.643,0.464,0.812,0.700,0.060,0.479,9.849,319.36
4,5,tfidf_word_logreg,tfidf,False,0.643,0.639,0.623,0.652,0.643,0.641,0.455,0.810,0.692,0.031,0.481,26.544,114.74
5,6,ft_xlm_roberta_base,finetuned,False,0.645,0.637,0.621,0.652,0.645,0.638,0.457,0.814,0.689,0.059,0.479,486.358,6636.63
6,7,ft_multilingual_bert,finetuned,False,0.634,0.624,0.607,0.639,0.633,0.624,0.438,0.804,0.682,0.099,0.498,403.567,7827.00
7,8,tfidf_word_complementnb,tfidf,False,0.618,0.615,0.599,0.630,0.618,0.618,0.418,0.795,0.678,0.140,0.533,3.231,102.40
8,9,ft_distilbert_multilingual,finetuned,False,0.621,0.609,0.593,0.624,0.620,0.608,0.417,0.792,0.664,0.094,0.514,259.479,3693.17
9,10,tfidf_word_multinomialnb,tfidf,False,0.615,0.606,0.591,0.622,0.614,0.602,0.405,0.784,0.662,0.056,0.508,4.011,181.64


**Is the best really better?** (paired McNemar with Holm correction + paired-bootstrap CI of the macro-F1 difference)

,method,macro_f1_diff,diff_ci_low,diff_ci_high,holm_p,verdict
0,majority_baseline,0.4631,0.4481,0.4786,0.0000,reference better
1,vader,0.3427,0.3214,0.3630,0.0000,reference better
2,vader_tuned,0.3420,0.3208,0.3622,0.0000,reference better
3,textblob,0.3588,0.3386,0.3786,0.0000,reference better
4,textblob_tuned,0.3569,0.3364,0.3765,0.0000,reference better
5,tfidf_word_logreg,0.0253,0.0145,0.0374,0.0000,reference better
6,tfidf_char_logreg,0.0132,0.0031,0.0231,0.0213,reference better
7,tfidf_word+char_linearsvc_calibrated,0.0153,0.0065,0.0243,0.1391,indistinguishable
8,tfidf_word_complementnb,0.0493,0.0347,0.0634,0.0000,reference better
9,tfidf_word_multinomialnb,0.0579,0.0434,0.0729,0.0000,reference better


PosixPath('/content/drive/MyDrive/chatlens_project/mlflow/mlflow.db')

In [10]:
run_task("sentiment_english_sentiment140", "sentiment140", "sentiment")
tracker.sync()

2026-11-03 13:45:38,456 INFO chatlens stage=sentiment_english_sentiment140:majority_baseline RUNNING (never run)
2026-11-03 13:45:39,289 INFO chatlens stage=sentiment_english_sentiment140:majority_baseline COMPLETE seconds=0.808 artifacts=2
2026-11-03 13:45:39,290 INFO chatlens stage=sentiment_english_sentiment140:vader RUNNING (never run)
2026-11-03 13:45:41,369 INFO chatlens stage=sentiment_english_sentiment140:vader COMPLETE seconds=2.052 artifacts=2
2026-11-03 13:45:41,372 INFO chatlens stage=sentiment_english_sentiment140:vader_tuned RUNNING (never run)
2026-11-03 13:45:50,173 INFO chatlens stage=sentiment_english_sentiment140:vader_tuned COMPLETE seconds=8.776 artifacts=2
2026-11-03 13:45:50,175 INFO chatlens stage=sentiment_english_sentiment140:textblob RUNNING (never run)
2026-11-03 13:45:54,896 INFO chatlens stage=sentiment_english_sentiment140:textblob COMPLETE seconds=4.693 artifacts=2
2026-11-03 13:45:54,898 INFO chatlens stage=sentiment_english_sentiment140:textblob_tuned 

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-11-03 14:01:35,485 INFO chatlens stage=sentiment_english_sentiment140:hf_xlmr_twitter_sentiment COMPLETE seconds=77.717 artifacts=2
2026-11-03 14:01:35,487 INFO chatlens stage=sentiment_english_sentiment140:hf_distilbert_multilingual_sentiment RUNNING (never run)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

2026-11-03 14:02:20,505 INFO chatlens stage=sentiment_english_sentiment140:hf_distilbert_multilingual_sentiment COMPLETE seconds=44.983 artifacts=2
2026-11-03 14:02:20,507 INFO chatlens stage=sentiment_english_sentiment140:hf_distilbert_sst2_english RUNNING (never run)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

... (9 outputs left out)

,rank,method,family,zero_shot,accuracy,macro_f1,macro_f1_ci_low,macro_f1_ci_high,weighted_f1,balanced_acc,mcc,roc_auc_ovr,pr_auc,ece,brier,fit_s,ms_per_1k
0,1,tfidf_word+char_logreg,tfidf,False,0.802,0.802,0.796,0.807,0.802,0.802,0.604,0.881,0.879,0.010,0.279,72.379,300.07
1,2,tfidf_word+char_sgd_modhuber,tfidf,False,0.802,0.802,0.796,0.807,0.802,0.802,0.604,0.882,0.877,0.042,0.283,54.085,315.80
2,3,tfidf_word+char_linearsvc_calibrated,tfidf,False,0.798,0.798,0.792,0.803,0.798,0.798,0.596,0.879,0.877,0.022,0.283,71.576,318.73
3,4,tfidf_word_logreg,tfidf,False,0.796,0.796,0.790,0.801,0.796,0.796,0.591,0.877,0.873,0.021,0.286,22.627,113.13
4,5,tfidf_char_logreg,tfidf,False,0.786,0.786,0.780,0.792,0.786,0.786,0.573,0.865,0.862,0.013,0.299,40.187,288.65
5,6,tfidf_word_multinomialnb,tfidf,False,0.786,0.786,0.780,0.791,0.786,0.786,0.573,0.869,0.866,0.078,0.310,18.897,193.53
6,7,tfidf_word_complementnb,tfidf,False,0.786,0.786,0.780,0.791,0.786,0.786,0.572,0.869,0.866,0.078,0.310,20.693,107.91
7,8,hf_xlmr_twitter_sentiment,pretrained,True,0.785,0.784,0.779,0.790,0.784,0.785,0.569,0.857,0.848,0.079,0.319,0.011,3943.31
8,9,tfidf_word_randomforest,tfidf,False,0.764,0.764,0.758,0.770,0.764,0.764,0.528,0.845,0.841,0.062,0.330,460.084,311.14
9,10,hf_distilbert_sst2_english,pretrained,True,0.709,0.706,0.699,0.712,0.706,0.709,0.427,0.790,0.782,0.255,0.533,0.009,1967.11


**Is the best really better?** (paired McNemar with Holm correction + paired-bootstrap CI of the macro-F1 difference)

,method,macro_f1_diff,diff_ci_low,diff_ci_high,holm_p,verdict
0,majority_baseline,0.4675,0.4612,0.4737,0.0000,reference better
1,vader,0.1566,0.1488,0.1645,0.0000,reference better
2,vader_tuned,0.1566,0.1488,0.1645,0.0000,reference better
3,textblob,0.2274,0.2194,0.2350,0.0000,reference better
4,textblob_tuned,0.2274,0.2194,0.2350,0.0000,reference better
5,tfidf_word_logreg,0.0062,0.0026,0.0101,0.0020,reference better
6,tfidf_char_logreg,0.0156,0.0116,0.0199,0.0000,reference better
7,tfidf_word+char_linearsvc_calibrated,0.0039,0.0016,0.0061,0.0020,reference better
8,tfidf_word_complementnb,0.0163,0.0117,0.0212,0.0000,reference better
9,tfidf_word_multinomialnb,0.0160,0.0113,0.0208,0.0000,reference better


PosixPath('/content/drive/MyDrive/chatlens_project/mlflow/mlflow.db')

## 8 · Emotion benchmark (English, 6 classes, official train/val/test)
Only models that output *emotions* are compared (lexicon polarity tools would be a different task, so they are excluded).

In [ ]:
run_task("emotion_english", "emotions", "emotion")
tracker.sync()

2026-11-03 14:03:43,855 INFO chatlens stage=emotion_english:majority_baseline RUNNING (never run)
2026-11-03 14:03:44,061 INFO chatlens stage=emotion_english:majority_baseline COMPLETE seconds=0.177 artifacts=2
2026-11-03 14:03:44,064 INFO chatlens stage=emotion_english:tfidf_word_logreg RUNNING (never run)
2026-11-03 14:04:17,155 INFO chatlens stage=emotion_english:tfidf_word_logreg COMPLETE seconds=33.063 artifacts=3
2026-11-03 14:04:17,157 INFO chatlens stage=emotion_english:tfidf_char_logreg RUNNING (never run)
2026-11-03 14:05:18,189 INFO chatlens stage=emotion_english:tfidf_char_logreg COMPLETE seconds=60.992 artifacts=3
2026-11-03 14:05:18,190 INFO chatlens stage=emotion_english:tfidf_word+char_logreg RUNNING (never run)
2026-11-03 14:06:34,660 INFO chatlens stage=emotion_english:tfidf_word+char_logreg COMPLETE seconds=76.442 artifacts=3
2026-11-03 14:06:34,661 INFO chatlens stage=emotion_english:tfidf_word+char_linearsvc_calibrated RUNNING (never run)
2026-11-03 14:07:05,272 IN

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

... (20 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3

## 9 · Topic modelling: six methods × k sweep (unsupervised: **no accuracy**)
LDA · NMF · K-means on TF-IDF · LSA+K-means · sentence-embedding K-means · BERTopic.
Metrics: NPMI **coherence** (interpretability proxy), topic **diversity**, and **silhouette / Davies-Bouldin /
Calinski-Harabasz** of the assignments measured in one shared space so methods are comparable. k is chosen by coherence
within a tolerance, ties broken by silhouette (coherence alone can reward a model that silently drops a theme).

In [ ]:
from chatlens.experiments import topic_benchmark
def _corpus():
    if _root("goemotions"):
        return "goemotions", load_goemotions_text(_root("goemotions"), CONFIG["TOPIC_MAX_DOCS"])["text"].tolist()
    # fallback: use the Roman Urdu training sentences (short informal text)
    return "roman_urdu", DATA["roman_urdu"]["splits"]["train"]["text"].tolist()
corpus_name, corpus_texts = _corpus()
ENCODER = None
if CONFIG["RUN_TRANSFORMERS"] and not SMOKE:
    try:
        from chatlens.nlp.topics import make_st_encoder
        ENCODER = make_st_encoder("paraphrase-multilingual-MiniLM-L12-v2")
    except Exception as e: print("sentence encoder unavailable:", str(e)[:150])
methods = [m for m in CONFIG["TOPIC_METHODS"] if not (SMOKE and m in ("embedding_kmeans", "bertopic"))]
TOPICS = topic_benchmark(corpus_texts, store, methods, tuple(CONFIG["TOPIC_KS"] if not SMOKE else (4, 6)), SEED, ENCODER,
                         CONFIG["TOPIC_MAX_DOCS"], corpus_name)
print(f"corpus: {corpus_name}, {TOPICS['n_docs']:,} usable documents")
if TOPICS["skipped"]: print("skipped:", TOPICS["skipped"])
store.save_csv("reports/tables/topics_sweep.csv", TOPICS["table"])
display(TOPICS["table"].round(3))
display(Markdown("**Selected k per method** (coherence within tolerance, then silhouette) and the topics themselves — *read these: metrics are proxies*"))
for m, row in TOPICS["best"].items():
    d = TOPICS["topics"][(m, int(row["k"]))]
    display(Markdown(f"**{m}** (k={int(row['k'])}, NPMI {row['npmi_coherence']:.3f}, diversity {row['topic_diversity']:.2f})"))
    for i, w in enumerate(d["topics"][:6], 1): print(f"   {i}. {', '.join(w[:8])}")

## 10 · Keyword extraction: six methods
TF-IDF · RAKE · YAKE · TextRank · embedding-MMR (KeyBERT-style) · frequency baseline. There are no gold keyphrases, so
*accuracy is not reported*; methods are compared by **bootstrap stability** (do keywords survive resampling?),
**agreement** (Jaccard) and runtime. Agreement shows methods are alike, not that they are right.

In [ ]:
from chatlens.experiments import keyword_benchmark
KW = keyword_benchmark(corpus_texts, store, ENCODER, 20, SEED, corpus_name, 10_000 if not SMOKE else 2000)
store.save_csv("reports/tables/keywords_summary.csv", KW["summary"])
display(KW["summary"].round(3))
agree = pd.Series(KW["raw"]["agreement_jaccard"]).sort_values(ascending=False)
display(Markdown("**Pairwise agreement (Jaccard of top-20)**")); display(agree.round(2).to_frame("jaccard").head(15))
if KW["raw"]["errors"]: print("not run:", KW["raw"]["errors"])

## 11 · Statistical analysis
*Why non-parametric tests?* Text lengths and response times are heavy-tailed (shown below), violating ANOVA's normality
assumption → Kruskal-Wallis / Mann-Whitney with rank-based effect sizes. We report **effect sizes and CIs** because with
thousands of rows even trivial differences are "significant" (statistical ≠ practical significance).

In [ ]:
from scipy import stats as ss
from chatlens import stats as cstats
rows, STAT_ROWS = [], []
for name, d in DATA.items():
    tr = d["splits"]["train"].copy(); tr["n_words"] = tr["text"].str.split().str.len()
    sample = np.log1p(tr["n_words"].sample(min(len(tr), 5000), random_state=SEED))
    nt = ss.normaltest(sample)
    kw_ = cstats.kruskal_by_group(tr["n_words"], tr["label"])
    counts = tr["label"].value_counts()
    chi = ss.chisquare(counts.values); w = float(np.sqrt(chi.statistic / counts.sum()))      # Cohen's w for goodness of fit
    r = {"dataset": name, "log-length skew": round(float(ss.skew(sample)), 2), "normality p (log-length)": f"{nt.pvalue:.2g}",
         "length differs by label (KW p)": f"{kw_.p_value:.3g}" if kw_ else "n/a",
         "KW effect (epsilon²)": f"{kw_.effect_size:.3f} [{kw_.magnitude}]" if kw_ else "n/a",
         "class balance χ² p": f"{chi.pvalue:.3g}", "balance effect (Cohen w)": round(w, 3)}
    STAT_ROWS.append(r)
display(pd.DataFrame(STAT_ROWS))
display(Markdown("Reading guide: a tiny p with a *negligible/small* effect means 'detectable but unimportant'."))
# model-comparison family-wise error control is already in the significance tables (Holm)
store.save_csv("reports/tables/dataset_statistics.csv", pd.DataFrame(STAT_ROWS))

## 12 · Export the best trained model + drift reference
The best **TF-IDF** model (CPU-friendly, ships inside the app/Docker image) is copied to `models/` with a model card.
The reference distribution of its test-set predictions is saved for drift monitoring (PSI) on new chats —
drift monitoring applies **only to trained models**, not to the rule-based parser.

In [ ]:
from chatlens.experiments import best_sklearn_method, export_model, reference_distribution
REF = {}
if "sentiment_roman_urdu" in TASKS and len(TASKS["sentiment_roman_urdu"]["table"]):
    t = TASKS["sentiment_roman_urdu"]; best_sk = best_sklearn_method(t["table"])
    p = export_model(store, "sentiment_roman_urdu", best_sk, ROOT / "models", "sentiment_roman_urdu.joblib",
                     t["metrics"][best_sk], DATA["roman_urdu"]["quality"]["fingerprint"])
    REF = reference_distribution(store, "sentiment_roman_urdu", best_sk)
    store.save_json("models/reference_distribution.json", REF)
    print("exported:", p, "\nreference distribution:", {k: round(v, 3) for k, v in REF.items()})
    print(open(str(Path(p).with_suffix(".card.json"))).read()[:600])
else:
    print("no Roman Urdu benchmark in this run: model export skipped")

## 13 · Analyse a WhatsApp chat
Your own export (set `CHAT_FILE`) or, if none is given, a **fabricated demo chat** whose message text is sampled from the
real corpora above. A fabricated chat's *temporal and behavioural* results only reflect the generator's settings.

In [ ]:
from chatlens.ingestion import read_chat
from chatlens.analyze import analyze_chat, NLPOptions
from chatlens.app.service import load_routers
from chatlens.synthetic import make_synthetic_chat, DEFAULT_PHRASES
SYNTHETIC = CONFIG["CHAT_FILE"] is None
if not SYNTHETIC:
    ing = read_chat(ROOT / "chats" / CONFIG["CHAT_FILE"]); chat_text = ing.text
    print(f"loaded {ing.n_bytes/1e6:.1f} MB ({ing.encoding}); content stays in this runtime/Drive")
else:
    bank = dict(DEFAULT_PHRASES)
    if "roman_urdu" in DATA:
        s = DATA["roman_urdu"]["splits"]["train"]; bank["roman"] = [t for t in s["text"].sample(300, random_state=SEED) if 3 <= len(t.split()) <= 25 and t.isascii()]
    if "emotions" in DATA:
        bank["en"] = DATA["emotions"]["splits"]["train"]["text"].sample(300, random_state=SEED).tolist()
    chat_text, truth = make_synthetic_chat(6000 if not SMOKE else 2500, seed=SEED, phrase_bank=bank)
    print("⚠️  FABRICATED chat generated (", truth.notes, ")")
router, desc = load_routers(ROOT / "models")
print("NLP routing:", desc)
reference = REF or None
t0 = time.time()
BUNDLE = analyze_chat(chat_text, anonymize_names=CONFIG["ANONYMIZE"],
                      nlp=NLPOptions(sentiment=router, topic_ks=(3, 5, 8), reference_sentiment_dist=reference, encoder=ENCODER))
print(f"analysed {BUNDLE.overview['total_messages']:,} messages in {time.time()-t0:.1f}s | chat type: {BUNDLE.meta['chat_type']} | quality ok: {BUNDLE.quality['ok']}")

In [ ]:
from chatlens.report import export_bundle
from chatlens.viz import save_all
from chatlens.app import render as R
stamp = time.strftime("%Y%m%d_%H%M%S"); OUT = ROOT / "reports" / "chat" / (("synthetic_" if SYNTHETIC else "chat_") + stamp)
files = export_bundle(BUNDLE, OUT, include_text=False, benchmark={k: v["table"].round(3) for k, v in TASKS.items() if len(v["table"])})
charts = save_all(BUNDLE, OUT / "charts")
display(Markdown(R.status_markdown(BUNDLE, desc, SYNTHETIC)))
display(R.quality_table(BUNDLE)); display(BUNDLE.participants.round(2).head(10))
display(Markdown(R.behaviour_markdown(BUNDLE))); display(Markdown(R.sentiment_markdown(BUNDLE))); display(Markdown(R.topics_markdown(BUNDLE)))
display(R.stats_table(BUNDLE))
mon = BUNDLE.nlp.get("sentiment", {}).get("monitoring", {})
print("NLP output monitoring:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in mon.items() if k != "predicted_distribution"})
print("\nsaved:", OUT, f"({len(charts['png'])} PNG + {len(charts['html'])} interactive HTML charts)")

In [ ]:
# a few charts inline
import matplotlib.image as mpimg
for name in ["activity_heatmap", "timeline", "participants", "response_times", "sentiment_participants"]:
    p = OUT / "charts" / "png" / f"{name}.png"
    if p.exists():
        plt.figure(figsize=(8, 4)); plt.imshow(mpimg.imread(p)); plt.axis("off"); plt.show()

## 14 · Final project report, manifest and deliverables bundle

In [ ]:
from chatlens.report import project_report
from chatlens.checkpoint import atomic_write_text
sections = [("Scope and honesty", "Data: public Kaggle corpora (not WhatsApp). Chat results come from "
             + ("a FABRICATED generator" if SYNTHETIC else "the user's own export (private; not included in this bundle)")
             + ". Transformer methods are present only if RUN_TRANSFORMERS was true and the run completed.")]
sections.append(("Datasets and quality", pd.DataFrame(rows)))
for task, out in TASKS.items():
    if len(out["table"]):
        sections += [(f"{task}: comparison", out["table"].round(3)), (f"{task}: significance vs best", out["significance"].round(4)),
                     (f"{task}: plots", ("images", [str(ROOT / f"reports/figures/{task}_macro_f1.png"), str(ROOT / f"reports/figures/{task}_confusion_best.png")]))]
        if out["errors"]: sections.append((f"{task}: methods that failed", {k: v for k, v in out["errors"].items()}))
sections += [("Topic modelling comparison", TOPICS["table"].round(3)), ("Keyword extraction comparison", KW["summary"].round(3)),
             ("Dataset statistics", pd.DataFrame(STAT_ROWS))]
prov = {**environment_info(), "seed": SEED, "mode": CONFIG["MODE"], "gpu": GPU, "tracking": tracker.backend,
        "datasets": {k: v.get("slug") or v.get("source") for k, v in DOWNLOADS.items()}}
atomic_write_text(ROOT / "reports" / "final_project_report.html", project_report("chatlens — project report", sections, prov))
tracker.sync()
display(pd.DataFrame(store.summary()))
print("final report:", ROOT / "reports" / "final_project_report.html")

In [ ]:
# Deliverables bundle: everything EXCEPT private chats and raw third-party datasets.
import zipfile
zp = ROOT / "deliverables" / f"chatlens_deliverables_{time.strftime('%Y%m%d_%H%M%S')}.zip"
skip_prefix = ("chats/", "data/raw/", "deliverables/", "nlp/ft_ckpt/")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(ROOT.rglob("*")):
        rel = f.relative_to(ROOT).as_posix()
        if f.is_file() and not rel.startswith(skip_prefix) and not rel.endswith((".tmp",)) and "mlruns" not in rel:
            z.write(f, rel)
print("bundle:", zp, f"({zp.stat().st_size/1e6:.1f} MB)")
print("\nWhere things are on Drive:")
for rel, what in [("reports/final_project_report.html", "final report"), ("reports/chat/", "chat report, tables, charts (HTML+PNG)"),
                  ("reports/tables/", "benchmark & statistics CSVs"), ("models/", "exported model + model card + reference distribution"),
                  ("nlp/", "per-method results, predictions, fitted models"), ("mlflow/", "experiment tracking DB + runs.jsonl"),
                  ("logs/", "run.log, environment.lock.txt"), ("manifest.json", "checkpoint manifest (stage status + SHA-256)")]:
    print(f"  {rel:38s} {what}")

## 15 · (Optional) Interactive dashboard in Colab
`LAUNCH_DASHBOARD=True` starts the Gradio app. **`share=True` creates a public internet URL** that routes through Gradio's
relay: anything uploaded there passes through a third-party service, so keep it `False` for private chats (use the
Colab proxy/port or run locally with Docker instead — see the README).

In [ ]:
if CONFIG["LAUNCH_DASHBOARD"]:
    os.environ["CHATLENS_MODEL_DIR"] = str(ROOT / "models")
    from chatlens.app.gradio_app import build_demo
    build_demo().launch(share=False, debug=False)     # change to share=True only if you accept the privacy trade-off
else:
    print("dashboard not launched (LAUNCH_DASHBOARD=False). Local: `pip install '.[app]' && chatlens-app`")

## Appendix · Resume, troubleshooting, updating the model
* **Disconnected?** Re-run from the top. `manifest.json` lists each stage; a stage is skipped only if its parameters are
  unchanged and every artifact still matches its SHA-256. A stage left `running` by a crash is re-run automatically.
* **Force a stage to recompute**: delete its entry from `manifest.json` (or change a parameter in the config cell).
* **Kaggle 403**: open the dataset page and click *Download/accept terms* once while logged in.
* **Out of GPU memory**: lower `FINETUNE_MAX_TRAIN`, or set `RUN_TRANSFORMERS=False` (all other stages still run).
* **Updating the production model**: re-run §7/§12 with new data → check the significance table (is it better than the
  current one?) → check ECE/calibration → export → bump the version → commit `models/` + the model card.